# CITE-seq to FCS

CITE-seq measures surface proteins of single cells with oligo-tagged antibodies (antibody-derived tags, ADT). Writing
the ADT counts to an FCS file allows gating the cells in flow cytometry software.

In [1]:
import os
import tempfile
import warnings

import anndata as ad
import flowio
import numpy as np
import pandas as pd

from anndata_fcs import read_fcs, write_fcs

# anndata<=0.13.4 reads a pandas option deprecated in pandas 3.1
warnings.filterwarnings("ignore", message="The 'future.infer_string' option is deprecated")

DATA_DIR = os.path.join(os.getcwd(), "..", "..", "test")
H5AD_FILENAME = os.path.join(DATA_DIR, "testing.h5ad")
OUT_DIR = tempfile.mkdtemp()

## Load CITE-seq data

In [2]:
adata = ad.read_h5ad(H5AD_FILENAME)
adata

AnnData object with n_obs × n_vars = 1000 × 32
    var: 'gene_ids', 'feature_types', 'genome', 'pattern', 'read', 'sequence'
    layers: None (.X)

In [3]:
adata.var.head()

,gene_ids,feature_types,genome,pattern,read,sequence
CD3_TotalSeqB,CD3,Antibody Capture,,^NNNNNNNNNN(BC)NNNNNNNNN,R2,CTCATTGTAACTCCT
CD4_TotalSeqB,CD4,Antibody Capture,,^NNNNNNNNNN(BC)NNNNNNNNN,R2,TGTTCCCGCTCAACT
CD8a_TotalSeqB,CD8a,Antibody Capture,,^NNNNNNNNNN(BC)NNNNNNNNN,R2,GCTGCGCTTTCCATT
CD11b_TotalSeqB,CD11b,Antibody Capture,,^NNNNNNNNNN(BC)NNNNNNNNN,R2,GACAAGTGATCTGCA
CD14_TotalSeqB,CD14,Antibody Capture,,^NNNNNNNNNN(BC)NNNNNNNNN,R2,TCTCAGACCTCCGTA


## Normalize

ADT counts are commonly normalized with the centered log ratio (CLR) transformation, here computed per cell. The counts are kept in `raw` and
`X` holds the CLR values.

In [4]:
counts = adata.X.toarray()
adata.raw = adata.copy()

log_counts = np.log1p(counts)
adata.X = log_counts - log_counts.mean(axis=1, keepdims=True)
adata.to_df().iloc[:5, :5]

,CD3_TotalSeqB,CD4_TotalSeqB,CD8a_TotalSeqB,CD11b_TotalSeqB,CD14_TotalSeqB
AAAGAACGTCGCTTGG-1,3.337880,-0.810532,3.524578,0.036766,-0.117385
TCGTGGGGTCATCCCT-1,3.630290,4.760497,-1.680450,-0.987303,-0.987303
GTTAGTGTCCAGCAAT-1,3.836781,4.526849,-0.561775,0.211415,-0.274093
TGTGGCGAGTCCCAGC-1,0.075997,2.417803,-0.552611,4.238000,4.181392
GCAACCGCAGGACATG-1,-0.543139,2.416966,-0.543139,4.610153,3.779668


## Write counts to FCS

`write_fcs` writes `raw` by default, i.e. the ADT counts. `var_names` are written as channel names (`$PnN`). Use
`pns_key` to write a column of `var` as marker labels (`$PnS`), which flow cytometry software shows next to the channel
name.

In [5]:
counts_fcs = os.path.join(OUT_DIR, "citeseq_counts.fcs")
write_fcs(adata, counts_fcs, pns_key="gene_ids")

/home/malte/Dokumente/Github/anndata-fcs/src/anndata_fcs/_io.py:62: FCSConversionWarning: X differs from raw and is not written. Use 'use_raw=False' to write X instead of raw.
  fcs_bytes = _anndata_to_fcs_bytes(


The FCS file can be read by any FCS reader, e.g. `flowio`.

In [6]:
fdata = flowio.FlowData(counts_fcs)
pd.DataFrame({"$PnN": fdata.pnn_labels, "$PnS": fdata.pns_labels}).tail()

,$PnN,$PnS
28,TIGIT_TotalSeqB,TIGIT
29,IgG1_control_TotalSeqB,IgG1
30,IgG2a_control_TotalSeqB,IgG2a
31,IgG2b_control_TotalSeqB,IgG2b
32,barcode_rank,


In [7]:
events = pd.DataFrame(fdata.as_array(preprocess=False), columns=fdata.pnn_labels)
events.iloc[:5, [0, 1, 2, -1]]

,CD3_TotalSeqB,CD4_TotalSeqB,CD8a_TotalSeqB,barcode_rank
0,189.0,2.0,228.0,0.0
1,404.0,1253.0,1.0,1.0
2,487.0,972.0,5.0,2.0
3,14.0,155.0,7.0,3.0
4,9.0,192.0,9.0,4.0


## Read FCS to AnnData

`read_fcs` restores the cell barcodes, `obs` and `var`. The `barcode_rank` channel is removed.

In [8]:
adata_counts = read_fcs(counts_fcs)
adata_counts

AnnData object with n_obs × n_vars = 1000 × 32
    var: 'gene_ids', 'feature_types', 'genome', 'pattern', 'read', 'sequence'
    uns: 'fcs'
    layers: None (.X)

In [9]:
adata_counts.obs.head()

""
AAAGAACGTCGCTTGG-1
TCGTGGGGTCATCCCT-1
GTTAGTGTCCAGCAAT-1
TGTGGCGAGTCCCAGC-1
GCAACCGCAGGACATG-1
